without text

In [ ]:
import cv2
import cvzone
from cvzone.FaceMeshModule import FaceMeshDetector
import numpy as np

cap = cv2.VideoCapture(0)
detector = FaceMeshDetector(maxFaces=1)

sen = 25  # more is less

while True:
    success, img = cap.read()
    img, faces = detector.findFaceMesh(img, draw=False)

    if faces:
        face = faces[0]
        pointLeft = face[145]
        pointRight = face[374]
        w, _ = detector.findDistance(pointLeft, pointRight)
        W = 6.3

        # Finding distance
        f = 840
        d = (W * f) / w
        print(d)

        cvzone.putTextRect(img, f'Depth: {int(d)}cm',
                           (face[10][0] - 100, face[10][1] - 50),
                           scale=2)

    cv2.imshow("Webcam Feed", img)
    cv2.waitKey(1)


80.80566568853749
81.88184086266999
81.88184086266999
81.69624504079421
80.46866190718563
81.69624504079421
81.69624504079421
81.88184086266999
81.69624504079421
81.88184086266999
83.15575746753798
81.88184086266999
81.88184086266999
81.88184086266999
81.69624504079421
81.88184086266999
81.88184086266999
81.88184086266999
81.69624504079421
81.69624504079421
81.88184086266999
81.69624504079421
81.88184086266999
81.88184086266999
81.88184086266999
81.88184086266999
81.88184086266999
81.88184086266999
81.88184086266999
82.0489769307614
81.88184086266999
81.88184086266999
81.88184086266999
80.64599838350142
80.80566568853749
80.64599838350142
83.33083325450524
80.64599838350142
81.88184086266999
82.0489769307614
81.88184086266999
81.88184086266999
82.0489769307614
80.80566568853749
82.0489769307614
82.0489769307614
82.0489769307614
80.64599838350142
82.0489769307614
82.19730301193994
82.19730301193994
82.19730301193994
82.0489769307614
80.94733898828326
80.80566568853749
82.19730301193994


With yolo for all objs

In [ ]:
import cv2
import cvzone
import numpy as np
import torch  # YOLOv5 requires PyTorch

# Load YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')  # Load YOLOv5 small model, you can change 'yolov5s' to 'yolov5m' or 'yolov5l' for larger models

cap = cv2.VideoCapture(0)

sen = 25  # more is less

while True:
    success, img = cap.read()
    results = model(img)  # Run YOLOv5 detection

    # Process results
    img = np.squeeze(results.render())  # Get the rendered image (with bounding boxes and labels)

    # If any objects are detected
    if results.xywh[0].size(0) > 0:
        for det in results.xywh[0]:
            x, y, w, h, conf, cls = det  # Get the bounding box (x, y, width, height), confidence, and class
            label = results.names[int(cls)]  # Get the object label

            # Assuming the real-world width of the object (W) and focal length (f) are known
            # For now, using a placeholder width of 6.3 cm for demonstration
            W = 6.3  # Real-world width of the object (in cm)
            f = 840  # Focal length (you can adjust based on your camera specs)
            d = (W * f) / w  # Calculate the distance to the object

            # Display the distance on the image
            cvzone.putTextRect(img, f'Distance: {int(d)}cm', (int(x), int(y) - 10), scale=2)

    cv2.imshow("Webcam Feed", img)
    cv2.waitKey(1)


Using cache found in C:\Users\Muhammad Haroon/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2024-11-7 Python-3.12.7 torch-2.2.2+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [4]:
import cv2
import torch
import numpy as np

# Load YOLOv5 pre-trained model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')  # You can use 'yolov5s', 'yolov5m', etc.

# Define real-world object sizes (width in cm or meters) for some common objects
# You may need to expand this for more objects
object_sizes = {
    'person': 50,    # average width of a person in cm
    'car': 180,      # average width of a car in cm
    'dog': 50,       # average width of a dog in cm
    'bicycle': 50,   # average width of a bicycle in cm
    # Add more objects here
}

# Load the image
img = cv2.imread("sample3.jpg")

# Run object detection
results = model(img)
# results.xywh contains the bounding boxes, classes, and confidences
boxes = results.xywh[0]

# Camera's focal length (can be calibrated for your specific camera)
focal_length = 840  # Adjust based on camera calibration

for box in boxes:
    x, y, w, h, conf, cls = box
    label = results.names[int(cls)]  # Get the object label (e.g., 'person', 'car', etc.)
    
    # Get the real-world width of the detected object
    if label in object_sizes:
        W = object_sizes[label]
    else:
        W = 100  # Default size for unknown objects (can adjust)

    # Calculate distance using the formula
    d = (W * focal_length) / w
    print(f"Object: {label}, Distance: {d} cm")

    # Draw bounding box and distance text
    cv2.rectangle(img, (int(x - w / 2), int(y - h / 2)), (int(x + w / 2), int(y + h / 2)), (0, 255, 0), 2)
    cv2.putText(img, f'{label}: {int(d)} cm', (int(x), int(y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Show the image with bounding boxes and distances
img = cv2.resize(img,(500,500))
cv2.imshow("Detected Objects", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


Using cache found in C:\Users\Muhammad Haroon/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2024-11-7 Python-3.12.7 torch-2.2.2+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Without YOLO with algo for any object

In [21]:
import cv2
import numpy as np

# Known parameters
W = 6.3  # Real-world width of the object in cm (adjust based on your reference object)
f = 840  # Focal length of the camera in pixels (calibrate for your camera)

# Load image
image_path = "sample6.jpg"  # Replace with your image path
img = cv2.imread(image_path)

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Apply GaussianBlur and Canny edge detection
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 50, 150)

# Find contours
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours:
    # Find the largest contour (assuming it's the object)
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Get bounding box around the largest contour
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Calculate the distance to the object
    d = (W * f) / w
    
    # Draw the bounding box and display the distance
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
    cv2.putText(img, f'Distance: {int(d)} cm', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
else:
    print("No object detected!")

# Show the result
img = cv2.resize(img, (500,500))
cv2.imshow("Object Detection", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


With web cam

In [22]:
import cv2
import numpy as np

# Known parameters
W = 6.3  # Real-world width of the object in cm (adjust based on your reference object)
f = 840  # Focal length of the camera in pixels (calibrate for your camera)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    if not success:
        print("Failed to capture image")
        break

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply GaussianBlur and Canny edge detection
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)

    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Find the largest contour (assuming it's the object)
        largest_contour = max(contours, key=cv2.contourArea)
        
        # Get bounding box around the largest contour
        x, y, w, h = cv2.boundingRect(largest_contour)
        
        # Calculate the distance to the object
        d = (W * f) / w
        
        # Draw the bounding box and display the distance
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, f'Distance: {int(d)} cm', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    else:
        cv2.putText(frame, "No object detected!", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Show the frame
    cv2.imshow("Webcam Object Detection", frame)

    # Exit loop on pressing 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()
